# Fretwork Generalization — Transformer Position Prior

Canonical **research / refine** harness for the two-tier position modeling strategy in [`docs/model.md`](../docs/model.md).

| Tier | Prior | Decoder | Role |
| :--- | :--- | :--- | :--- |
| **Production** | GuitarSet empirical unigram | Viterbi (`combined_all_tuned`) | Baseline to beat |
| **Research** | DadaGP Transformer (~1M) | Beam search (width 8) | Long-range fingering prior |

**Data synergy:** `dadagp_distilled/` (structure) · GuitarSet (precision / GT) · GAPS (generalization).

**Default mode:** load existing `tab_transformer_final.pt` (no retrain). Set `RUN_TRAIN = True` only when you intentionally rebuild weights.

**Runtime:** PyTorch recommended for the research path. Production baseline uses `backend/fretboard.py` (numpy only).


## 1. Setup & Dependencies

Install required libraries in the current Jupyter environment if they are missing.


In [ ]:
# ── Setup & Dependencies ──────────────────────────────────────────────────────
# Installs and validates necessary packages (torch, jams) in the notebook environment.
# Automatically reinstalls torch with CUDA support if a CPU-only version is detected.

# Import system libraries for environment inspection and subprocess execution.
import sys
import subprocess

try:
    # Attempt to import PyTorch and JAMS libraries.
    import torch
    import jams
    # Check if the current torch version is CPU-bound or CUDA is unavailable.
    if "+cpu" in torch.__version__ or not torch.cuda.is_available():
        # Raise an import error to trigger the package reinstallation logic.
        raise ImportError("CUDA is not available with the current PyTorch installation.")
    # Log that dependencies are already correctly installed with active CUDA.
    print(f"Dependencies already satisfied: torch {torch.__version__} (CUDA active).")
except ImportError as e:
    # Log that PyTorch setup failed and recovery installation is beginning.
    print(f"Target PyTorch environment check failed: {e}")
    print("Installing/updating dependencies with CUDA-enabled PyTorch...")
    # Remove existing conflicting or CPU-bound PyTorch packages.
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "torch", "torchvision", "torchaudio", "-y"])
    try:
        # Attempt to install PyTorch using the CUDA 12.4 wheel repository.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "--index-url", "https://download.pytorch.org/whl/cu124"])
    except subprocess.CalledProcessError:
        # Fall back to installing PyTorch using the CUDA 12.1 wheel repository if 12.4 fails.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "--index-url", "https://download.pytorch.org/whl/cu121"])
    # Install the jams library to parse dataset JAMS annotations.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jams"])
    # Alert the user to restart the Jupyter kernel for settings to apply.
    print("\nInstallation complete! IMPORTANT: Please restart the Jupyter kernel (Kernel -> Restart) to load the CUDA-enabled PyTorch.")


## 1.1 Hardware Acceleration Diagnostics

Check if PyTorch is utilizing the CPU or GPU, and display detailed information about the active GPU.


In [ ]:
# ── Hardware Acceleration Diagnostics ─────────────────────────────────────────
# Verifies PyTorch's execution device and prints diagnostic information about the GPU.

# Import torch to perform hardware capability checks.
import torch

# Determine whether CUDA or CPU is active based on PyTorch configuration.
device_name = "CUDA" if torch.cuda.is_available() else "CPU"
print(f"Active Device: {device_name}")

# Print detailed GPU specifications if CUDA is available.
if torch.cuda.is_available():
    # Retrieve the name of the current NVIDIA GPU device.
    gpu_name = torch.cuda.get_device_name(0)
    # Fetch the compute capability version of the active GPU.
    compute_capability = torch.cuda.get_device_capability(0)
    # Calculate the total memory available on the GPU in gigabytes.
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    
    # Print the compiled GPU specifications to the console.
    print(f"GPU Model: {gpu_name}")
    print(f"Compute Capability: {compute_capability[0]}.{compute_capability[1]}")
    print(f"Total GPU Memory: {total_memory_gb:.2f} GB")
else:
    # Log a fallback message indicating CPU execution.
    print("Warning: Running on CPU. PyTorch cannot access GPU acceleration.")


## 2. Configuration & paths

Local monorepo roots (not Colab Drive). Edit flags below for subset size and train gate.


In [ ]:
# ── Configuration & paths ──────────────────────────────────────────────────────
# Sets default configurations, constants, file paths, and execution flags.
from __future__ import annotations

import json
import math
import sys
import time
import random
import gzip
from pathlib import Path
from collections import defaultdict
from itertools import product

import numpy as np
import pandas as pd

# ── Repo root resolution ─────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "jupyter_notebooks":
    CAPSTONE_ROOT = NOTEBOOK_DIR.parent
else:
    CAPSTONE_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "backend").exists() else NOTEBOOK_DIR.parent

if str(CAPSTONE_ROOT) not in sys.path:
    sys.path.insert(0, str(CAPSTONE_ROOT))

# ── Artifact paths ───────────────────────────────────────────────────────────
DISTILL_DIR = CAPSTONE_ROOT / "dadagp_distilled"
TP_DIR = CAPSTONE_ROOT / "transformer_prior"
TP_DIR.mkdir(parents=True, exist_ok=True)

TRANSFORMER_PRIOR_PATH = CAPSTONE_ROOT / "tab_transformer_final.pt"
if not TRANSFORMER_PRIOR_PATH.exists():
    TRANSFORMER_PRIOR_PATH = TP_DIR / "tab_transformer_final.pt"

GUITARSET_PRIOR_PATH = CAPSTONE_ROOT / "guitarset_position_prior.json"
GUITARSET_JAMS_DIR = CAPSTONE_ROOT / "FullGuitarSetData" / "JamsFiles"
GUITARSET_AUDIO_DIR = CAPSTONE_ROOT / "FullGuitarSetData" / "AudioFiles"
GAPS_DIR = CAPSTONE_ROOT / "GAPS_dataset"

WINDOWS_PATH = TP_DIR / "training_windows.npz"
CKPT_PATH = TP_DIR / "tab_transformer_ckpt.pt"
EXPORT_PATH = TP_DIR / "tab_transformer_final.pt"
COUNT_PRIOR_PATH = CAPSTONE_ROOT / "dadagp_position_prior.json.gz"

OUTPUT_DIR = CAPSTONE_ROOT / "outputs" / "fretwork_generalization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Run flags ────────────────────────────────────────────────────────────────
RUN_TRAIN = False
RUN_INTRINSIC = True
RUN_GUITARSET_ORACLE = True
RUN_GAPS = True
RUN_REFINE_SWEEP = False
MAX_EVAL_TRACKS = 360
BEAM_WIDTH = 8
TRANSFORMER_WEIGHT = 2.0
SEED = 0

# ── Fretboard / model constants ──────────────────────────────────────────────
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
STRING_NAMES = ["low_E", "A", "D", "G", "B", "high_E"]
MAX_FRET = 24
MAX_FRET_MODEL = 24
MIN_MIDI, MAX_MIDI = 40, 88
N_PITCH = MAX_MIDI - MIN_MIDI + 1
N_POS = 6 * (MAX_FRET_MODEL + 1)
BOS_POS = N_POS
POS_VOCAB = N_POS + 1
CTX = 64
STRIDE = 32
VAL_TRACK_FRAC = 0.10
MAX_WINDOWS = 2_000_000
D_MODEL, N_LAYERS, N_HEADS = 128, 4, 4
BATCH, LR, TRAIN_STEPS = 256, 3e-4, 12_000
ONSET_TOLERANCE_SECONDS = 0.035
MAX_GROUP_CANDIDATES = 25
COMFORTABLE_SPAN = 5
MAX_REACHABLE_SPAN = 7
HAND_MOVE_WEIGHT = 0.70

print("CAPSTONE_ROOT:", CAPSTONE_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Artifact probe

Reports what is present so missing data fails loudly with an action, not a silent crash later.


In [ ]:
# ── Artifact Probe ───────────────────────────────────────────────────────────
# Scans and validates presence of raw/compiled data files needed for evaluation.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


def probe_artifacts():
    rows = []
    checks = [
        ("dadagp_distilled shards", DISTILL_DIR, lambda p: p.is_dir() and any(p.glob("shard_*.jsonl.gz"))),
        ("tab_transformer_final.pt", TRANSFORMER_PRIOR_PATH, lambda p: p.is_file()),
        ("guitarset_position_prior.json", GUITARSET_PRIOR_PATH, lambda p: p.is_file()),
        ("GuitarSet JAMS", GUITARSET_JAMS_DIR, lambda p: p.is_dir() and any(p.glob("*.jams"))),
        ("GuitarSet audio", GUITARSET_AUDIO_DIR, lambda p: p.is_dir() and any(p.rglob("*.wav"))),
        ("GAPS audio", GAPS_DIR / "audio", lambda p: p.is_dir() and any(p.glob("*.wav"))),
        (
            "GAPS scores (MIDI/XML)",
            GAPS_DIR,
            lambda p: p.is_dir()
            and (
                any(p.rglob("*.mid"))
                or any(p.rglob("*.midi"))
                or any(p.rglob("*.xml"))
                or any(p.rglob("*.musicxml"))
            ),
        ),
        ("training windows npz", WINDOWS_PATH, lambda p: p.is_file()),
    ]
    for name, path, ok_fn in checks:
        try:
            ok = bool(ok_fn(path))
            note = "ready" if ok else "MISSING"
        except Exception as e:
            ok, note = False, str(e)
        rows.append({"artifact": name, "path": str(path), "ok": ok, "note": note})
    df = pd.DataFrame(rows)
    display(df)
    return df


artifact_df = probe_artifacts()
HAS_TRANSFORMER_WEIGHTS = bool(TRANSFORMER_PRIOR_PATH.exists())
HAS_GUITARSET = GUITARSET_JAMS_DIR.is_dir() and any(GUITARSET_JAMS_DIR.glob("*.jams"))
HAS_GAPS_SCORES = False
if GAPS_DIR.exists():
    HAS_GAPS_SCORES = any(GAPS_DIR.rglob("*.mid")) or any(GAPS_DIR.rglob("*.midi")) or any(
        GAPS_DIR.rglob("*.xml")
    ) or any(GAPS_DIR.rglob("*.musicxml"))
print(
    "HAS_TRANSFORMER_WEIGHTS:",
    HAS_TRANSFORMER_WEIGHTS,
    "| HAS_GUITARSET:",
    HAS_GUITARSET,
    "| HAS_GAPS_SCORES:",
    HAS_GAPS_SCORES,
)


## 3. Backend production baseline + empirical prior

Loads the GuitarSet unigram into `backend.fretboard` and exposes `assign_combined_all_tuned` as the production decoder.


In [ ]:
# ── Backend production baseline ──────────────────────────────────────────────
# Sets up Viterbi decoders and GuitarSet unigram count priors as baseline metrics.
import backend.fretboard as fb
from backend.jams_processor import process_jams_file

if GUITARSET_PRIOR_PATH.exists():
    with open(GUITARSET_PRIOR_PATH, encoding="utf-8") as f:
        raw_prior = json.load(f)
    loaded = {tuple(int(x) for x in k.split(",")): float(v) for k, v in raw_prior.items()}
    fb.POSITION_PRIOR_COSTS = loaded
    print(f"Loaded GuitarSet unigram prior into backend.fretboard: {len(loaded)} entries")
else:
    print("WARNING: guitarset_position_prior.json missing — production prior is empty (default costs).")


def jams_to_notes(jams_path: Path):
    """Oracle pitches + true string/fret from GuitarSet JAMS."""
    rec = process_jams_file(jams_path)
    notes = []
    for n in rec.get("notes", []):
        midi = int(round(float(n["midi"])))
        notes.append(
            {
                "start": float(n["start"]),
                "duration": float(n.get("duration", 0.0)),
                "midi": midi,
                "pitch_class": midi % 12,
                "true_string": n.get("true_string"),
                "true_fret": n.get("true_fret"),
            }
        )
    return notes, rec


def assign_production_viterbi(notes):
    """Tier-1 production path."""
    return fb.assign_combined_all_tuned(notes)


print("Production baseline ready: assign_combined_all_tuned")


## 4. Research model — TabTransformer load

Causal transformer: $P(\text{string},\text{fret} \mid \text{pitch}, \text{history})$. Invalid positions masked.


In [ ]:
# ── Research model — TabTransformer load ──────────────────────────────────────
# Defines the causal TabTransformer architecture and implements weight loading.
TORCH_OK = False
DEVICE = "cpu"
TP_MODEL = None
_TP_CTX = CTX
_VALID_T = None
TabTransformer = None
masked_logits = None
load_transformer = None

try:
    import torch
    import torch.nn as nn

    TORCH_OK = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("torch:", torch.__version__, "| device:", DEVICE)
except ImportError:
    print("PyTorch not installed — research beam decoder disabled. Install torch to enable Tier 2.")
    torch = None
    nn = None

if TORCH_OK:

    class TabTransformer(nn.Module):
        """Causal transformer: predicts each note's (string, fret) from pitch, prev pos, history."""

        def __init__(self, d=D_MODEL, layers=N_LAYERS, heads=N_HEADS, ctx=CTX, multitask=False):
            super().__init__()
            self.ctx = ctx
            self.multitask = multitask
            self.emb_pitch = nn.Embedding(N_PITCH, d)
            self.emb_prev = nn.Embedding(POS_VOCAB, d)
            self.emb_flag = nn.Embedding(2, d)
            self.emb_time = nn.Embedding(ctx, d)
            layer = nn.TransformerEncoderLayer(
                d_model=d,
                nhead=heads,
                dim_feedforward=4 * d,
                dropout=0.1,
                batch_first=True,
                norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
            if self.multitask:
                self.string_head = nn.Linear(d, 6)
                self.fret_head = nn.Linear(d, MAX_FRET_MODEL + 1)
            else:
                self.head = nn.Linear(d, N_POS)

        def forward(self, pitch, prev_pos, flag):
            B, T = pitch.shape
            t_idx = torch.arange(T, device=pitch.device).unsqueeze(0).expand(B, T)
            x = (
                self.emb_pitch(pitch)
                + self.emb_prev(prev_pos)
                + self.emb_flag(flag)
                + self.emb_time(t_idx)
            )
            causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=pitch.device), 1)
            h = self.encoder(x, mask=causal)
            if self.multitask:
                return self.string_head(h), self.fret_head(h)
            return self.head(h)

    VALID_POS = np.zeros((N_PITCH, N_POS), dtype=bool)
    for s, om in enumerate(OPEN_STRING_MIDI):
        for f in range(MAX_FRET_MODEL + 1):
            m = om + f
            if MIN_MIDI <= m <= MAX_MIDI:
                VALID_POS[m - MIN_MIDI, s * 25 + f] = True
    _VALID_T = torch.tensor(VALID_POS, dtype=torch.bool, device=DEVICE)

    def masked_logits(logits, pitch):
        return logits.masked_fill(~_VALID_T[pitch], -1e9)

    def load_transformer(path: Path):
        global TP_MODEL, _TP_CTX
        ck = torch.load(path, map_location=DEVICE, weights_only=False)
        cfg = ck.get("config") or {"d": D_MODEL, "layers": N_LAYERS, "heads": N_HEADS, "ctx": CTX, "multitask": False}
        model = TabTransformer(
            cfg["d"], cfg["layers"], cfg["heads"], cfg["ctx"], multitask=cfg.get("multitask", False)
        ).to(DEVICE)
        model.load_state_dict(ck["model"])
        model.eval()
        TP_MODEL = model
        _TP_CTX = int(cfg["ctx"])
        n_params = sum(t.numel() for t in model.parameters()) / 1e6
        print(f"Loaded transformer from {path} | ctx={_TP_CTX} | {n_params:.2f}M params")
        return model

    if HAS_TRANSFORMER_WEIGHTS:
        load_transformer(TRANSFORMER_PRIOR_PATH)
    else:
        print(
            "No transformer weights found — set RUN_TRAIN=True or place tab_transformer_final.pt at repo root."
        )


## 5. Optional retrain (gated)

Skipped unless `RUN_TRAIN = True`. Flow: distillate → windows → train → export.


In [ ]:
# ── Optional retrain (gated) ──────────────────────────────────────────────────
# Processes distillate shards into fixed-length windows and trains the model.
def track_to_arrays(events):
    pitches, poss, flags = [], [], []
    for item in events:
        if item[0] != "g":
            continue
        group = sorted(item[1], key=lambda sf: OPEN_STRING_MIDI[sf[0]] + sf[1])
        for j, (s, f) in enumerate(group):
            m = OPEN_STRING_MIDI[s] + f
            if not (MIN_MIDI <= m <= MAX_MIDI) or not (0 <= f <= MAX_FRET_MODEL):
                return None
            pitches.append(m - MIN_MIDI)
            poss.append(s * 25 + f)
            flags.append(1 if j == 0 else 0)
    return (
        np.array(pitches, np.uint8),
        np.array(poss, np.uint8),
        np.array(flags, np.uint8),
    )


def prepare_windows():
    if WINDOWS_PATH.exists():
        print("Windows already prepared — skipping rebuild.")
        return
    shard_files = sorted(DISTILL_DIR.glob("shard_*.jsonl.gz"))
    if not shard_files:
        raise FileNotFoundError(f"No shards in {DISTILL_DIR}")
    rng = random.Random(SEED)
    train_w, val_w = [], []
    t0 = time.time()
    for fp in shard_files:
        with gzip.open(fp, "rt", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                arrs = track_to_arrays(rec["events"])
                if arrs is None or len(arrs[0]) < CTX + 1:
                    continue
                dest = val_w if rng.random() < VAL_TRACK_FRAC else train_w
                p, q, g = arrs
                for st in range(0, len(p) - CTX, STRIDE):
                    dest.append((p[st : st + CTX], q[st : st + CTX], g[st : st + CTX]))
        print(f"{fp.name}: {len(train_w):,} train / {len(val_w):,} val ({time.time() - t0:.0f}s)")
        if len(train_w) >= MAX_WINDOWS:
            break
    rng.shuffle(train_w)
    train_w = train_w[:MAX_WINDOWS]

    def stack(ws):
        return (
            np.stack([w[0] for w in ws]),
            np.stack([w[1] for w in ws]),
            np.stack([w[2] for w in ws]),
        )

    tr = stack(train_w)
    if val_w:
        va = stack(val_w[:20_000])
    else:
        va = (np.zeros((0, CTX), np.uint8), np.zeros((0, CTX), np.uint8), np.zeros((0, CTX), np.uint8))
    np.savez_compressed(
        WINDOWS_PATH,
        train_pitch=tr[0],
        train_pos=tr[1],
        train_flag=tr[2],
        val_pitch=va[0],
        val_pos=va[1],
        val_flag=va[2],
    )
    print(f"Saved windows -> {WINDOWS_PATH}")


class DistilledDataset(torch.utils.data.Dataset):
    def __init__(self, pitch, pos, flag):
        self.pitch = pitch
        self.pos = pos
        self.flag = flag
        
    def __len__(self):
        return len(self.pitch)
        
    def __getitem__(self, idx):
        p = torch.tensor(self.pitch[idx], dtype=torch.long)
        q = torch.tensor(self.pos[idx], dtype=torch.long)
        g = torch.tensor(self.flag[idx], dtype=torch.long)
        prev = torch.cat([torch.tensor([BOS_POS], dtype=torch.long), q[:-1]])
        return p, prev, g, q


def train_transformer(steps=TRAIN_STEPS):
    if not TORCH_OK:
        raise RuntimeError("torch required for training")
    prepare_windows()
    z = np.load(WINDOWS_PATH)
    
    # Seeding
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        
    train_ds = DistilledDataset(z["train_pitch"], z["train_pos"], z["train_flag"])
    val_ds = DistilledDataset(z["val_pitch"], z["val_pos"], z["val_flag"])
    
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True
    )
    
    model = TabTransformer(multitask=False).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    
    # OneCycle LR Scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, steps_per_epoch=steps, epochs=1, pct_start=0.1
    )
    
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    
    start_step = 0
    best_val_loss = float("inf")
    
    if CKPT_PATH.exists():
        ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        start_step = ck["step"]
        if "best_val_loss" in ck:
            best_val_loss = ck["best_val_loss"]
        print(f"Resumed from step {start_step} with best_val_loss {best_val_loss:.4f}")
        
    train_iter = iter(train_loader)
    model.train()
    t0 = time.time()
    
    for step in range(start_step + 1, steps + 1):
        try:
            p, prev, g, q = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            p, prev, g, q = next(train_iter)
            
        p, prev, g, q = p.to(DEVICE), prev.to(DEVICE), g.to(DEVICE), q.to(DEVICE)
        
        opt.zero_grad()
        
        # Mixed Precision Autocast
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            if getattr(model, "multitask", False):
                str_logits, fr_logits = model(p, prev, g)
                target_str = q // 25
                target_fr = q % 25
                loss_str = loss_fn(str_logits.reshape(-1, 6), target_str.reshape(-1))
                loss_fr = loss_fn(fr_logits.reshape(-1, 25), target_fr.reshape(-1))
                loss = 0.6 * loss_str + 0.4 * loss_fr
            else:
                logits = masked_logits(model(p, prev, g), p)
                loss = loss_fn(logits.reshape(-1, N_POS), q.reshape(-1))
            
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
        scheduler.step()
        
        if step % 500 == 0:
            # Eval validation loss
            model.eval()
            val_loss = 0.0
            val_count = 0
            with torch.no_grad():
                for vp, vprev, vg, vq in val_loader:
                    vp, vprev, vg, vq = vp.to(DEVICE), vprev.to(DEVICE), vg.to(DEVICE), vq.to(DEVICE)
                    with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                        if getattr(model, "multitask", False):
                            vstr_logits, vfr_logits = model(vp, vprev, vg)
                            vtarget_str = vq // 25
                            vtarget_fr = vq % 25
                            vloss_str = loss_fn(vstr_logits.reshape(-1, 6), vtarget_str.reshape(-1))
                            vloss_fr = loss_fn(vfr_logits.reshape(-1, 25), vtarget_fr.reshape(-1))
                            vloss = 0.6 * vloss_str + 0.4 * vloss_fr
                        else:
                            vlogits = masked_logits(model(vp, vprev, vg), vp)
                            vloss = loss_fn(vlogits.reshape(-1, N_POS), vq.reshape(-1))
                    val_loss += vloss.item() * vp.size(0)
                    val_count += vp.size(0)
            mean_val_loss = val_loss / max(1, val_count)
            model.train()
            
            print(f"step {step} | loss {loss.item():.3f} | val_loss {mean_val_loss:.3f} | {time.time() - t0:.0f}s")
            
            if mean_val_loss < best_val_loss:
                best_val_loss = mean_val_loss
                best_ckpt_path = TP_DIR / "best_transformer_ckpt.pt"
                torch.save(
                    {
                        "model": model.state_dict(),
                        "opt": opt.state_dict(),
                        "step": step,
                        "best_val_loss": best_val_loss,
                    },
                    best_ckpt_path,
                )
                print(f"New best val loss! Saved best checkpoint -> {best_ckpt_path}")
                
        if step % 1000 == 0:
            torch.save(
                {
                    "model": model.state_dict(),
                    "opt": opt.state_dict(),
                    "step": step,
                    "best_val_loss": best_val_loss,
                },
                CKPT_PATH,
            )
            
    payload = {
        "model": model.state_dict(),
        "config": {"d": D_MODEL, "layers": N_LAYERS, "heads": N_HEADS, "ctx": CTX, "multitask": getattr(model, "multitask", False)},
    }
    torch.save(payload, EXPORT_PATH)
    root_export = CAPSTONE_ROOT / "tab_transformer_final.pt"
    torch.save(payload, root_export)
    print("exported:", EXPORT_PATH, "and", root_export)
    load_transformer(EXPORT_PATH)


if RUN_TRAIN:
    train_transformer()
else:
    print("RUN_TRAIN=False — using existing checkpoint if loaded.")


## 6. Intrinsic eval (held-out DadaGP windows)

Compares transformer top-1 when windows exist. Scores second half of each window (warm context).


In [ ]:
# ── Intrinsic eval (held-out DadaGP windows) ──────────────────────────────────
# Evaluates TabTransformer next-position prediction accuracy on validation windows.
intrinsic_results = {}


def run_intrinsic_eval(n_eval_windows=500):
    if not TORCH_OK or TP_MODEL is None:
        print("Skip intrinsic: no torch model.")
        return {}
    if not WINDOWS_PATH.exists():
        print("Skip intrinsic: no training_windows.npz (set RUN_TRAIN once or copy windows).")
        return {}
    z = np.load(WINDOWS_PATH)
    VAL = (z["val_pitch"], z["val_pos"], z["val_flag"])
    if VAL[0].shape[0] == 0:
        print("Skip intrinsic: empty val split.")
        return {}
    n = min(n_eval_windows, VAL[0].shape[0])
    idx = np.arange(n)
    p = torch.tensor(VAL[0][idx], dtype=torch.long, device=DEVICE)
    q = torch.tensor(VAL[1][idx], dtype=torch.long, device=DEVICE)
    prev = torch.cat(
        [torch.full((q.shape[0], 1), BOS_POS, dtype=torch.long, device=DEVICE), q[:, :-1]],
        dim=1,
    )
    g = torch.tensor(VAL[2][idx], dtype=torch.long, device=DEVICE)
    TP_MODEL.eval()
    with torch.no_grad():
        pred = masked_logits(TP_MODEL(p, prev, g), p).argmax(-1).cpu().numpy()
    q_np = VAL[1][idx]
    half = CTX // 2
    correct = (pred[:, half:] == q_np[:, half:]).sum()
    total = pred[:, half:].size
    acc = float(correct) / float(total)
    out = {"transformer_top1": acc, "n_notes": int(total), "n_windows": int(n)}
    print(f"Intrinsic transformer top-1 (warm half): {acc:.3f} on {total:,} notes / {n} windows")
    return out


if RUN_INTRINSIC:
    intrinsic_results = run_intrinsic_eval()
else:
    print("RUN_INTRINSIC=False")


## 7. Research beam decoder (history-conditioned)

Exact Viterbi does not apply when the prior depends on full decoded history. Beam width defaults to 8; hybrid cost = playability/unigram base + `TRANSFORMER_WEIGHT * NLL`.


In [ ]:
# ── Research beam decoder (history-conditioned) ───────────────────────────────
# Implements beam search decoder that integrates the history-conditioned prior.
def get_possible_positions(midi_note, max_fret=MAX_FRET):
    midi_note = int(round(midi_note))
    out = []
    # Dynamic open string MIDI from backend.fretboard
    for s, om in enumerate(fb.OPEN_STRING_MIDI):
        f = midi_note - om
        if 0 <= f <= max_fret:
            out.append({"string": s, "fret": f, "midi": midi_note})
    return out


def group_notes_by_onset(notes, tolerance=ONSET_TOLERANCE_SECONDS):
    if not notes:
        return []
    # Avoid GT-informed sorting sequence for cleaner decoder evaluation
    notes_sorted = sorted(
        notes,
        key=lambda x: (
            x["start"],
            x["midi"],
        ),
    )
    groups, current = [], [notes_sorted[0]]
    group_start = notes_sorted[0]["start"]
    for n in notes_sorted[1:]:
        if abs(n["start"] - group_start) <= tolerance:
            current.append(n)
        else:
            groups.append(current)
            current = [n]
            group_start = n["start"]
    groups.append(current)
    return groups


def group_span(frets):
    fretted = [f for f in frets if f > 0]
    if not fretted:
        return 0
    return max(fretted) - min(fretted)


def group_playability_cost(positions):
    if not positions:
        return 0.0
    strings = [p["string"] for p in positions]
    frets = [p["fret"] for p in positions]
    if len(strings) != len(set(strings)):
        return float("inf")
    cost = 0.0
    span = group_span(frets)
    if span > COMFORTABLE_SPAN:
        cost += 2.0 * (span - COMFORTABLE_SPAN)
    if span > MAX_REACHABLE_SPAN:
        cost += 25.0 * (span - MAX_REACHABLE_SPAN)
    return cost


def position_prior_cost_local(midi, position):
    key = (int(midi), int(position["string"]), int(position["fret"]))
    return float(fb.POSITION_PRIOR_COSTS.get(key, 0.75))


def candidate_groups_hybrid(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    """Lean candidate pool: playability + GuitarSet unigram."""
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n["midi"])
        if not pos:
            return []
        position_lists.append(pos)
    space = 1
    for pl in position_lists:
        space *= len(pl)
    candidates = []
    if space > 20_000:
        used = set()
        pick = []
        for i in sorted(range(len(group_notes)), key=lambda k: -group_notes[k]["midi"]):
            opts = sorted(position_lists[i], key=lambda p: (p["fret"], p["string"]))
            chosen = next((p for p in opts if p["string"] not in used), opts[0])
            used.add(chosen["string"])
            pick.append((i, chosen))
        pick.sort(key=lambda x: x[0])
        combo = [p for _, p in pick]
        prior = float(
            np.mean([position_prior_cost_local(n["midi"], p) for n, p in zip(group_notes, combo)])
        )
        base = group_playability_cost(combo) + prior
        return [{"positions": combo, "base_cost": float(base if math.isfinite(base) else 100.0)}]

    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p["string"] for p in combo}) != len(combo):
            continue
        play = group_playability_cost(combo)
        if not math.isfinite(play):
            continue
        prior = float(
            np.mean([position_prior_cost_local(n["midi"], p) for n, p in zip(group_notes, combo)])
        )
        base = play + prior
        candidates.append({"positions": combo, "base_cost": float(base)})
    candidates.sort(key=lambda c: c["base_cost"])
    return candidates[:max_candidates]


def get_valid_mask(open_strings, device=DEVICE):
    valid_pos = np.zeros((N_PITCH, N_POS), dtype=bool)
    for s, om in enumerate(open_strings):
        for f in range(MAX_FRET_MODEL + 1):
            m = om + f
            if MIN_MIDI <= m <= MAX_MIDI:
                valid_pos[m - MIN_MIDI, s * 25 + f] = True
    return torch.tensor(valid_pos, dtype=torch.bool, device=device)


def _score_extensions(histories, extensions, valid_mask):
    """Transformer NLL for candidate extensions under each beam history."""
    seq_p, seq_prev, seq_g, meta = [], [], [], []
    for b, (hp, hq, hg) in enumerate(histories):
        for c, (ep, eq, eg) in enumerate(extensions[b]):
            p = (hp + ep)[-_TP_CTX:]
            q = (hq + eq)[-_TP_CTX:]
            g = (hg + eg)[-_TP_CTX:]
            prev = [BOS_POS] + q[:-1]
            seq_p.append(p)
            seq_prev.append(prev)
            seq_g.append(g)
            meta.append((b, c, len(ep), len(p)))
    maxlen = max(len(s) for s in seq_p)

    def pad(seqs, val):
        return torch.tensor(
            [s + [val] * (maxlen - len(s)) for s in seqs], dtype=torch.long, device=DEVICE
        )

    P, PR, G = pad(seq_p, 0), pad(seq_prev, BOS_POS), pad(seq_g, 0)
    with torch.no_grad():
        logits = TP_MODEL(P, PR, G)
        logits = logits.masked_fill(~valid_mask[P], -1e9)
        logp = torch.log_softmax(logits, dim=-1)
    out = defaultdict(dict)
    for row, (b, c, n_new, L) in enumerate(meta):
        nll = 0.0
        full_q = (histories[b][1] + list(extensions[b][c][1]))[-_TP_CTX:]
        for t in range(L - n_new, L):
            nll -= float(logp[row, t, full_q[t]])
        out[b][c] = nll
    return out


def assign_beam_transformer(notes, transformer_weight=TRANSFORMER_WEIGHT, beam_width=BEAM_WIDTH, tuning="standard", capo=0):
    if not TORCH_OK or TP_MODEL is None:
        raise RuntimeError("Transformer not loaded")
    with fb.fretboard_config(tuning=tuning, capo=capo):
        valid_mask = get_valid_mask(fb.OPEN_STRING_MIDI, device=DEVICE)
        groups = group_notes_by_onset(notes)
        if not groups:
            return []
        beams = [{"hp": [], "hq": [], "hg": [], "cost": 0.0, "center": None, "choice": []}]
        allc = []
        for g in groups:
            cands = candidate_groups_hybrid(g)
            allc.append(cands if cands else None)

        for gi, g in enumerate(groups):
            cands = allc[gi]
            if cands is None:
                for b in beams:
                    b["choice"].append(None)
                continue
            exts_per_cand = []
            for c in cands:
                pos_sorted = sorted(c["positions"], key=lambda p_: p_["midi"])
                ep = [int(p_["midi"]) - MIN_MIDI for p_ in pos_sorted]
                eq = [int(p_["string"]) * 25 + int(p_["fret"]) for p_ in pos_sorted]
                eg = [1] + [0] * (len(pos_sorted) - 1)
                exts_per_cand.append((ep, eq, eg))
            histories = [(b["hp"], b["hq"], b["hg"]) for b in beams]
            extensions = [exts_per_cand for _ in beams]
            nll = _score_extensions(histories, extensions, valid_mask)

            scored = []
            for bi, b in enumerate(beams):
                for ci, c in enumerate(cands):
                    cf = [p_["fret"] for p_ in c["positions"] if p_["fret"] > 0]
                    cc = float(np.mean(cf)) if cf else 0.0
                    move = 0.0
                    if b["center"] is not None and cf:
                        move = HAND_MOVE_WEIGHT * abs(cc - b["center"])
                    total = b["cost"] + c["base_cost"] + move + transformer_weight * nll[bi][ci]
                    scored.append((total, bi, ci, cc if cf else b["center"]))
            scored.sort(key=lambda x: x[0])
            new_beams = []
            for total, bi, ci, center in scored[:beam_width]:
                b = beams[bi]
                ep, eq, eg = exts_per_cand[ci]
                new_beams.append(
                    {
                        "hp": (b["hp"] + ep)[-_TP_CTX:],
                        "hq": (b["hq"] + eq)[-_TP_CTX:],
                        "hg": (b["hg"] + eg)[-_TP_CTX:],
                        "cost": total,
                        "center": center,
                        "choice": b["choice"] + [ci],
                    }
                )
            beams = new_beams

        best = min(beams, key=lambda b: b["cost"])
        out = []
        for gi, (g, ci) in enumerate(zip(groups, best["choice"])):
            if ci is None or allc[gi] is None:
                continue
            c = allc[gi][ci]
            pos_sorted = sorted(c["positions"], key=lambda p_: p_["midi"])
            g_sorted = sorted(g, key=lambda n_: n_["midi"])
            for note, p_ in zip(g_sorted, pos_sorted):
                row = dict(note)
                row.update(
                    {
                        "pred_string": p_["string"],
                        "pred_fret": p_["fret"],
                        "method": "beam_transformer",
                    }
                )
                out.append(row)
        return sorted(out, key=lambda x: (x["start"], x["midi"]))


print(
    "Beam decoder ready"
    if TORCH_OK and TP_MODEL is not None
    else "Beam decoder NOT ready (missing torch or weights)"
)


## 8. GuitarSet oracle assignment eval

**Oracle pitches** from JAMS (no Basic Pitch) → isolate fret assignment. Metrics: exact position, string-only, fret-only, thinner-string error rate, G–B confusions.


In [ ]:
# ── GuitarSet oracle assignment eval ──────────────────────────────────────────
# Compares production Viterbi and research beam transformer decoders on GuitarSet ground truth.
def score_predictions(pred_rows):
    """Exact / string / fret accuracy vs true_* fields."""
    n = exact = str_ok = fret_ok = 0
    thinner_pred = 0
    gb_conf = 0
    for r in pred_rows:
        ts, tf = r.get("true_string"), r.get("true_fret")
        ps, pf = r.get("pred_string"), r.get("pred_fret")
        if ts is None or tf is None or ps is None or pf is None:
            continue
        ts, tf, ps, pf = int(ts), int(tf), int(ps), int(pf)
        n += 1
        if ps == ts and pf == tf:
            exact += 1
        if ps == ts:
            str_ok += 1
        if pf == tf:
            fret_ok += 1
        if ps > ts:
            thinner_pred += 1
        if {ts, ps} == {2, 4} and ts != ps:
            gb_conf += 1
    if n == 0:
        return {"n": 0}
    return {
        "n": n,
        "exact_position_acc": exact / n,
        "string_acc": str_ok / n,
        "fret_acc": fret_ok / n,
        "thinner_string_error_rate": thinner_pred / n,
        "gb_confusion_rate": gb_conf / n,
    }


def list_eval_jams(max_tracks=MAX_EVAL_TRACKS, seed=SEED):
    files = sorted(GUITARSET_JAMS_DIR.glob("*.jams"))
    rng = random.Random(seed)
    rng.shuffle(files)
    return files[:max_tracks]


def run_guitarset_oracle(max_tracks=MAX_EVAL_TRACKS):
    if not HAS_GUITARSET:
        print("Skip GuitarSet oracle: JAMS not found.")
        return pd.DataFrame(), pd.DataFrame()
    files = list_eval_jams(max_tracks)
    rows = []
    failed_count = 0
    for jp in files:
        notes, _ = jams_to_notes(jp)
        if len(notes) < 4:
            continue
        try:
            t_start = time.time()
            prod = assign_production_viterbi(notes)
            t_prod = time.time() - t_start
            m_prod = score_predictions(prod)
            m_prod.update({"track": jp.stem, "method": "combined_all_tuned", "decode_time_sec": t_prod})
            rows.append(m_prod)
        except Exception as e:
            failed_count += 1
            rows.append(
                {"track": jp.stem, "method": "combined_all_tuned", "error": str(e), "n": 0}
            )
        if TORCH_OK and TP_MODEL is not None:
            try:
                t_start = time.time()
                research = assign_beam_transformer(notes)
                t_res = time.time() - t_start
                m_r = score_predictions(research)
                m_r.update({"track": jp.stem, "method": "beam_transformer", "decode_time_sec": t_res})
                rows.append(m_r)
            except Exception as e:
                failed_count += 1
                rows.append(
                    {"track": jp.stem, "method": "beam_transformer", "error": str(e), "n": 0}
                )
        print(f"scored {jp.stem}: notes={len(notes)}")
    
    print(f"GuitarSet oracle run complete. Failed runs: {failed_count}")
    df = pd.DataFrame(rows)
    if not df.empty and "exact_position_acc" in df.columns:
        summary = (
            df.dropna(subset=["exact_position_acc"])
            .groupby("method")
            .agg(
                tracks=("track", "count"),
                notes=("n", "sum"),
                exact_position_acc=("exact_position_acc", "mean"),
                exact_position_std=("exact_position_acc", "std"),
                string_acc=("string_acc", "mean"),
                fret_acc=("fret_acc", "mean"),
                thinner_err=("thinner_string_error_rate", "mean"),
                gb_conf=("gb_confusion_rate", "mean"),
                avg_decode_time=("decode_time_sec", "mean"),
            )
            .reset_index()
        )
        print("\n=== GuitarSet oracle summary (mean over tracks) ===")
        display(summary)
        
        # Paired Wilcoxon signed-rank test
        try:
            prod_accs = df[df["method"] == "combined_all_tuned"].sort_values("track")["exact_position_acc"].values
            res_accs = df[df["method"] == "beam_transformer"].sort_values("track")["exact_position_acc"].values
            if len(prod_accs) == len(res_accs) and len(prod_accs) >= 5:
                from scipy.stats import wilcoxon
                stat, p_val = wilcoxon(res_accs, prod_accs)
                print(f"Wilcoxon signed-rank test p-value: {p_val:.4f} (p < 0.05 is significant)")
        except Exception as e:
            print("Could not calculate statistical significance test:", e)
            
        out_path = OUTPUT_DIR / "guitarset_oracle_by_track.csv"
        df.to_csv(out_path, index=False)
        summary.to_csv(OUTPUT_DIR / "guitarset_oracle_summary.csv", index=False)
        print("Saved:", out_path)
        return df, summary
    display(df)
    return df, pd.DataFrame()


guitarset_df, guitarset_summary = pd.DataFrame(), pd.DataFrame()
if RUN_GUITARSET_ORACLE:
    guitarset_df, guitarset_summary = run_guitarset_oracle()
else:
    print("RUN_GUITARSET_ORACLE=False")


## 9. GAPS generalization (skip-safe)

When aligned MIDI/MusicXML with string/fret tags are present, evaluate the same decoders on non-studio material. Local trees that only contain `audio/` skip with an actionable message.


In [ ]:
# ── GAPS generalization (skip-safe) ───────────────────────────────────────────
# Parses score-aligned MIDI and MusicXML files to evaluate transcription generalization on non-studio audio.
import xml.etree.ElementTree as ET
import mido

gaps_results = {"status": "not_run"}


def parse_musicxml_guitar(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    notes = []
    
    for part in root.findall(".//part"):
        for measure in part.findall(".//measure"):
            for note_elem in measure.findall(".//note"):
                if note_elem.find("rest") is not None:
                    continue
                pitch_elem = note_elem.find("pitch")
                if pitch_elem is None:
                    continue
                step = pitch_elem.find("step").text
                octave = int(pitch_elem.find("octave").text)
                alter_elem = pitch_elem.find("alter")
                alter = int(alter_elem.text) if alter_elem is not None else 0
                
                pc_map = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}
                midi = 12 * (octave + 1) + pc_map[step] + alter
                
                fret = None
                string = None
                tech = note_elem.find(".//technical")
                if tech is not None:
                    fret_elem = tech.find("fret")
                    string_elem = tech.find("string")
                    if fret_elem is not None:
                        fret = int(fret_elem.text)
                    if string_elem is not None:
                        string = 6 - int(string_elem.text)
                
                if fret is not None and string is not None:
                    notes.append({
                        "midi": midi,
                        "string": string,
                        "fret": fret
                    })
    return notes


def load_midi_notes(midi_path):
    mid = mido.MidiFile(midi_path)
    active_notes = {}
    notes = []
    current_time = 0.0
    for msg in mid:
        current_time += msg.time
        if msg.type == 'note_on' and msg.velocity > 0:
            active_notes[(msg.channel, msg.note)] = current_time
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            key = (msg.channel, msg.note)
            if key in active_notes:
                start_time = active_notes.pop(key)
                duration = current_time - start_time
                notes.append({
                    "start": start_time,
                    "duration": duration,
                    "midi": msg.note,
                })
    notes.sort(key=lambda x: x["start"])
    return notes


def run_gaps_section(max_tracks=MAX_EVAL_TRACKS):
    if not GAPS_DIR.exists():
        return {
            "status": "missing_gaps_dir",
            "action": "snapshot_download xavriley/GAPS -> GAPS_dataset/",
        }
    audio_dir = GAPS_DIR / "audio"
    xml_dir = GAPS_DIR / "musicxml"
    midi_dir = GAPS_DIR / "midi"
    if not (audio_dir.is_dir() and xml_dir.is_dir() and midi_dir.is_dir()):
        return {
            "status": "gaps_subdirs_missing",
            "action": "Check GAPS dataset snapshot download structure.",
        }
        
    wav_files = sorted(audio_dir.glob("*.wav"))
    if not wav_files:
        return {"status": "no_gaps_audio_files"}
        
    eval_wavs = wav_files[:max_tracks]
    rows = []
    print(f"Running GAPS generalization evaluation on {len(eval_wavs)} tracks...")
    
    for wav_path in eval_wavs:
        stem = wav_path.stem
        xml_path = xml_dir / f"{stem}.xml"
        midi_path = midi_dir / f"{stem}.mid"
        if not xml_path.is_file():
            xml_path = next(xml_dir.glob(f"{stem}*.xml"), None)
        if not midi_path.is_file():
            midi_path = next(midi_dir.glob(f"{stem}*.mid"), None)
            
        if not xml_path or not midi_path:
            print(f"Skipping track {stem}: xml or midi missing.")
            continue
            
        try:
            xml_notes = parse_musicxml_guitar(xml_path)
            midi_notes = load_midi_notes(midi_path)
            
            xml_by_pitch = defaultdict(list)
            for xn in xml_notes:
                xml_by_pitch[xn["midi"]].append(xn)
                
            midi_by_pitch = defaultdict(list)
            for mn in midi_notes:
                midi_by_pitch[mn["midi"]].append(mn)
                
            notes = []
            for pitch, m_list in midi_by_pitch.items():
                x_list = xml_by_pitch.get(pitch, [])
                for mn, xn in zip(m_list, x_list):
                    notes.append({
                        "start": mn["start"],
                        "duration": mn["duration"],
                        "midi": pitch,
                        "true_string": xn["string"],
                        "true_fret": xn["fret"]
                    })
            notes.sort(key=lambda x: x["start"])
            
            if len(notes) < 4:
                continue
                
            # Run production Viterbi
            try:
                prod = assign_production_viterbi(notes)
                m_prod = score_predictions(prod)
                m_prod.update({"track": stem, "method": "combined_all_tuned"})
                rows.append(m_prod)
            except Exception as e:
                rows.append({"track": stem, "method": "combined_all_tuned", "error": str(e), "n": 0})
                
            # Run research beam transformer
            if TORCH_OK and TP_MODEL is not None:
                try:
                    research = assign_beam_transformer(notes)
                    m_r = score_predictions(research)
                    m_r.update({"track": stem, "method": "beam_transformer"})
                    rows.append(m_r)
                except Exception as e:
                    rows.append({"track": stem, "method": "beam_transformer", "error": str(e), "n": 0})
                    
            print(f"scored gaps track {stem}: notes={len(notes)}")
        except Exception as e:
            print(f"Failed to process gaps track {stem}: {e}")
            
    df = pd.DataFrame(rows)
    if not df.empty and "exact_position_acc" in df.columns:
        summary = (
            df.dropna(subset=["exact_position_acc"])
            .groupby("method")
            .agg(
                tracks=("track", "count"),
                notes=("n", "sum"),
                exact_position_acc=("exact_position_acc", "mean"),
                string_acc=("string_acc", "mean"),
                fret_acc=("fret_acc", "mean"),
                thinner_err=("thinner_string_error_rate", "mean"),
                gb_conf=("gb_confusion_rate", "mean"),
            )
            .reset_index()
        )
        print("\n=== GAPS generalization summary (mean over tracks) ===")
        display(summary)
        summary.to_csv(OUTPUT_DIR / "gaps_generalization_summary.csv", index=False)
        return {
            "status": "success",
            "n_tracks": len(eval_wavs),
            "summary": summary.to_dict(orient="records"),
        }
    return {"status": "no_data_processed"}


if RUN_GAPS:
    gaps_results = run_gaps_section()
    print(json.dumps(gaps_results, indent=2))
else:
    print("RUN_GAPS=False")


## 10. Refine sweep (optional)

Small grid over `TRANSFORMER_WEIGHT` × `BEAM_WIDTH` on a tiny GuitarSet subset. Enable with `RUN_REFINE_SWEEP = True`.


In [ ]:
# ── Refine sweep (optional) ───────────────────────────────────────────────────
# Conducts a hyperparameter grid search (transformer weights vs beam widths) on a subset of recordings.
refine_df = pd.DataFrame()


def run_refine_sweep(
    weights=(1.25, 1.75, 2.0, 2.5),
    beams=(4, 8),
    max_tracks=4,
):
    if not (TORCH_OK and TP_MODEL is not None and HAS_GUITARSET):
        print("Skip refine sweep: need torch model + GuitarSet.")
        return pd.DataFrame()
    files = list_eval_jams(max_tracks)
    rows = []
    for w in weights:
        for b in beams:
            accs = []
            for jp in files:
                notes, _ = jams_to_notes(jp)
                if len(notes) < 4:
                    continue
                pred = assign_beam_transformer(notes, transformer_weight=w, beam_width=b)
                m = score_predictions(pred)
                if m.get("n", 0):
                    accs.append(m["exact_position_acc"])
            rows.append(
                {
                    "transformer_weight": w,
                    "beam_width": b,
                    "mean_exact_position_acc": float(np.mean(accs)) if accs else None,
                    "tracks": len(accs),
                }
            )
            print(f"W={w} beam={b} -> {rows[-1]['mean_exact_position_acc']}")
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "refine_sweep.csv", index=False)
    display(df)
    return df


if RUN_REFINE_SWEEP:
    refine_df = run_refine_sweep()
else:
    print("RUN_REFINE_SWEEP=False — enable after baseline oracle looks healthy.")


## 11. Report — write metrics for `docs/model.md`

Aggregates this session into `outputs/fretwork_generalization/report.json`.


In [ ]:
# ── Report — write metrics for docs/model.md ──────────────────────────────────
# Consolidates configuration flags, loaded structures, and validation accuracy metrics into a JSON report.
report = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "capstone_root": str(CAPSTONE_ROOT),
    "flags": {
        "RUN_TRAIN": RUN_TRAIN,
        "RUN_INTRINSIC": RUN_INTRINSIC,
        "RUN_GUITARSET_ORACLE": RUN_GUITARSET_ORACLE,
        "RUN_GAPS": RUN_GAPS,
        "RUN_REFINE_SWEEP": RUN_REFINE_SWEEP,
        "MAX_EVAL_TRACKS": MAX_EVAL_TRACKS,
        "BEAM_WIDTH": BEAM_WIDTH,
        "TRANSFORMER_WEIGHT": TRANSFORMER_WEIGHT,
    },
    "artifacts": artifact_df.to_dict(orient="records")
    if isinstance(artifact_df, pd.DataFrame)
    else [],
    "torch_ok": TORCH_OK,
    "transformer_loaded": TP_MODEL is not None,
    "intrinsic": intrinsic_results,
    "gaps": gaps_results,
    "guitarset_summary": guitarset_summary.to_dict(orient="records")
    if isinstance(guitarset_summary, pd.DataFrame) and not guitarset_summary.empty
    else [],
    "promotion_reminder": [
        "Head-to-head GuitarSet win vs combined_all_tuned",
        "Thinner-string / G-B residual improvement",
        "GAPS non-regression when scores available",
        "Latency acceptable on ECS",
        "Difficulty / tuning / capo parity",
    ],
}

run_id = time.strftime("%Y%m%d_%H%M%S")
report_path = OUTPUT_DIR / "report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("Wrote", report_path)

history_dir = OUTPUT_DIR / "history"
history_dir.mkdir(parents=True, exist_ok=True)
hist_path = history_dir / f"report_{run_id}.json"
with open(hist_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("Wrote historical backup", hist_path)

print(
    json.dumps(
        {k: report[k] for k in ("torch_ok", "transformer_loaded", "intrinsic", "gaps")},
        indent=2,
    )
)
if report["guitarset_summary"]:
    print("GuitarSet summary:")
    print(pd.DataFrame(report["guitarset_summary"]).to_string(index=False))


## Next steps

1. Install PyTorch in the notebook kernel if the beam path is disabled.
2. Raise `MAX_EVAL_TRACKS` for a fuller held-out comparison.
3. Complete GAPS score download + MusicXML string/fret parser when status is `pairs_found_parse_pending`.
4. Enable `RUN_REFINE_SWEEP` and freeze best `(TRANSFORMER_WEIGHT, BEAM_WIDTH)` into [`docs/model.md`](../docs/model.md).
5. Only then consider production promotion criteria in model.md §6.

Donors / legacy: `Transformer_Position_Prior.ipynb`, `Fretwork_Transformer_test*.ipynb`, `AudioToTab_VariantEval_v1.ipynb`.
